# Trabajo final · Del NodoBot al preprint

**Métodos Numéricos · UNAL Manizales · octubre–noviembre de 2026 · Juan Pablo Esteban Ojeda**

En tu parcial, NodoBot le ganó a Chebyshev con los datos de Colombia y perdió con los de Perú. Tú concluiste que había aprendido una propiedad *de los datos* y no *de los nodos*. En este trabajo vas a convertir esa intuición en un resultado medido sobre **216 países**, con una cota que se puede demostrar y calcular, y lo vas a escribir como artículo.

El cuaderno va en el mismo orden que la guía:

| Pieza | Semana | Celdas |
|---|---|---|
| A · La forma baricéntrica | 1 | A1–A3 |
| B · El rompecabezas de Lebesgue | 2 | B1–B3 |
| C · NodoBot de verdad | 3 | C1–C5 |
| E · AAA (opcional) | 4 | E1–E2 |

Ritual de siempre: **predice en tu guía → corre la celda → explica la diferencia**. Solo cambias lo marcado con `# <-- CAMBIA ESTO`.

### Fuente de los datos
- **Indicador:** población total (SP.POP.TOTL), Banco Mundial, *World Development Indicators*. Página: https://data.worldbank.org/indicator/SP.POP.TOTL
- **Archivo que descarga el cuaderno:** https://raw.githubusercontent.com/datasets/population/main/data/population.csv (paquete *datasets/population*, licencia PDDL 1.0; el dato original del Banco Mundial es CC BY 4.0). Es el mismo archivo del parcial.
- **Países:** los 216 países con dato completo de 1960 a 2020. Se excluyen los agregados del Banco Mundial (Mundo, regiones, grupos de ingreso).
- **Copia congelada:** `datos/poblacion_paises_1960_2020.csv` en el repositorio del curso, tomada el 27 de octubre de 2026. Si la descarga falla o los valores cambiaron, el cuaderno usa la copia, para que tus cifras coincidan con las de la guía.

In [ ]:
# Celda 0 · Cargar los datos de 216 países (URL pública + copia congelada)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, time, warnings

URL = "https://raw.githubusercontent.com/datasets/population/main/data/population.csv"
COPIA = "https://raw.githubusercontent.com/cmorregof/methods_exam/main/trabajo-final-nodobot/datos/poblacion_paises_1960_2020.csv"
AGREGADOS = set('''AFE AFW ARB CEB CSS EAP EAR EAS ECA ECS EMU EUU FCS HIC HPC IBD IBT IDA IDB IDX INX LAC
LCN LDC LIC LMC LMY LTE MEA MIC MNA NAC OED OSS PRE PSS PST SAS SSA SSF SST TEA TEC TLA TMN TSA TSS UMC WLD'''.split())
ANOS = np.arange(1960, 2021).astype(float)          # 61 años: la "verdad"

def tabla_paises(df):
    df = df[~df["Country Code"].isin(AGREGADOS) & (df["Year"] <= 2020)]
    datos, nombres = {}, {}
    for cod, g in df.groupby("Country Code"):
        s_ = g.set_index("Year")["Value"]
        v = np.array([s_.get(int(a), np.nan) for a in ANOS], dtype=float)
        if np.isfinite(v).all():
            datos[cod] = v / 1e6                           # en MILLONES
            nombres[cod] = g["Country Name"].iloc[0]
    return datos, nombres

try:
    congelada, _ = tabla_paises(pd.read_csv(COPIA))
except Exception as e:
    congelada = None
    print("No se pudo leer la copia congelada (", type(e).__name__, ").")
try:
    DATOS, NOMBRES = tabla_paises(pd.read_csv(URL))
    if congelada is not None and (set(DATOS) != set(congelada) or
                                  any(not np.allclose(DATOS[c], congelada[c]) for c in DATOS)):
        print("OJO: la URL trae valores distintos a la copia del 27 oct 2026. Usamos la copia.")
        DATOS, NOMBRES = tabla_paises(pd.read_csv(COPIA))
    else:
        print("Datos descargados de la URL.")
except Exception as e:
    print("No se pudo descargar la URL (", type(e).__name__, "). Usamos la copia congelada.")
    DATOS, NOMBRES = tabla_paises(pd.read_csv(COPIA))

PAISES = sorted(DATOS)
COL, PER = DATOS["COL"], DATOS["PER"]
print("Países:", len(PAISES), "  Colombia 1960:", COL[0], "M   2020:", COL[-1], "M")

**Celda 1 · Herramientas.** No las modifiques, pero léelas: son las fórmulas de la guía.

- `s(año)`: el cambio de variable s = (año − 1990)/30, que lleva [1960, 2020] a [−1, 1].
- `pesos(xn)`: los pesos baricéntricos wⱼ = 1 / ∏_{k≠j} (xⱼ − xₖ) (en la variable s).
- `bary(xn, yn, t)`: la **segunda forma baricéntrica** (pieza A).
- `lebesgue(xn, t)`: la función de Lebesgue λ(t) = Σ |Lⱼ(t)|; su máximo es Λ.
- `error(nodos, y)`: interpola con esos años y devuelve |P(año) − dato| en los 61 años.
- `piso(y, n)`: el **piso minimax discreto** E: el menor error máximo posible en los 61 años con **cualquier** polinomio de grado ≤ n (pieza B). Se calcula con un problema de programación lineal: no necesitas saber resolverlo, sí saber qué calcula.
- `equi(k)`, `cheb(k)`, `mock(k)`: tres reglas que **no miran los datos**.
- `voraz(y, k)`: la regla de AAA trasladada a polinomios: empieza con 1960 y 2020 y agrega el año de mayor error.
- `busqueda_local(y, k, ...)`: NodoBot: parte de Chebyshev y mueve un año ±1 mientras baje el error.

In [ ]:
# Celda 1 · Herramientas
from scipy.optimize import linprog

def s(ano):
    return (np.asarray(ano, dtype=float) - 1990) / 30

def pesos(xn):
    x = s(xn)
    return np.array([1 / np.prod(x[j] - np.delete(x, j)) for j in range(len(x))])

def bary(xn, yn, t):
    '''Segunda forma baricéntrica: P(t) = sum(w_j y_j/(t-x_j)) / sum(w_j/(t-x_j)).'''
    x, tt, w = s(xn), np.atleast_1d(s(t)), pesos(xn)
    D = tt[:, None] - x[None, :]
    en_nodo = np.isclose(D, 0); D[en_nodo] = 1
    C = w / D
    P = (C @ np.asarray(yn, float)) / C.sum(axis=1)
    fila, col = np.where(en_nodo); P[fila] = np.asarray(yn, float)[col]   # en un nodo, P = dato
    return P

def lebesgue(xn, t):
    '''lambda(t) = sum |L_j(t)|, usando L_j(t) = (w_j/(t-x_j)) / sum_k (w_k/(t-x_k)).'''
    x, tt, w = s(xn), np.atleast_1d(s(t)), pesos(xn)
    D = tt[:, None] - x[None, :]
    en_nodo = np.isclose(D, 0); D[en_nodo] = 1
    C = w / D
    lam = np.abs(C).sum(axis=1) / np.abs(C.sum(axis=1))
    lam[en_nodo.any(axis=1)] = 1.0
    return lam

T_FINA = np.linspace(1960, 2020, 6001)
def Lambda(nodos): return lebesgue(np.array(sorted(nodos), float), T_FINA).max()

def error(nodos, y, t=ANOS):
    xn = np.array(sorted(nodos), dtype=float)
    yn = y[(xn - 1960).astype(int)]
    return np.abs(bary(xn, yn, t) - y)

def piso(y, n):
    '''min sobre p de grado <= n de max_{años} |y - p|  (programación lineal).'''
    V = np.polynomial.chebyshev.chebvander(s(ANOS), n); m, k = V.shape
    c = np.zeros(k + 1); c[-1] = 1                      # minimizar e
    A = np.vstack([np.hstack([V, -np.ones((m, 1))]), np.hstack([-V, -np.ones((m, 1))])])
    r = linprog(c, A_ub=A, b_ub=np.concatenate([y, -y]),
                bounds=[(None, None)] * k + [(0, None)], method="highs")
    return r.x[-1]

def equi(k):
    return ANOS[np.round(np.linspace(0, 60, k)).astype(int)]

def cheb(k):
    i = np.arange(k); c = 1990 + 30 * np.cos((2 * i + 1) * np.pi / (2 * k))
    return np.sort(np.array([ANOS[np.argmin(np.abs(ANOS - v))] for v in c]))

def mock(k):
    '''mock-Chebyshev: los años más cercanos a los k nodos de Chebyshev-Lobatto (incluyen 1960 y 2020).'''
    i = np.arange(k); c = 1990 + 30 * np.cos(np.pi * i / (k - 1)); out = []
    for v in c:
        for j in np.argsort(np.abs(ANOS - v)):
            if ANOS[j] not in out:
                out.append(ANOS[j]); break
    return np.sort(np.array(out))

def voraz(y, k):
    S = [1960.0, 2020.0]
    while len(S) < k:
        e = error(S, y); e[(np.array(S) - 1960).astype(int)] = -1
        S = sorted(S + [ANOS[e.argmax()]])
    return np.array(S)

def busqueda_local(y, k, inicio=None, Lmax=np.inf, mascara=None):
    '''NodoBot: mueve un año +-1 mientras baje el error máximo medido en los años de `mascara`.
    Solo acepta conjuntos con constante de Lebesgue <= Lmax (el verificador).'''
    mascara = np.ones(len(ANOS), bool) if mascara is None else mascara
    T_GRUESA = np.linspace(1960, 2020, 1201)
    S = list(cheb(k) if inicio is None else inicio)
    mejor = error(S, y)[mascara].max()
    while True:
        cand = None
        for p in range(k):
            for d in (-1, 1):
                nuevo = S[p] + d
                if nuevo < 1960 or nuevo > 2020 or nuevo in S:
                    continue
                prueba = sorted(S[:p] + [nuevo] + S[p + 1:])
                e = error(prueba, y)[mascara].max()
                if e < mejor and (cand is None or e < cand[0]):
                    if Lmax == np.inf or lebesgue(np.array(prueba), T_GRUESA).max() <= Lmax:
                        cand = (e, prueba)
        if cand is None:
            return np.array(S), mejor
        mejor, S = cand

NODOBOT = [1960, 1962, 1966, 1970, 1978, 1983, 1989, 1991, 1999, 2005, 2011, 2015, 2020]
print("Listo. Chebyshev con k = 13:", cheb(13).astype(int))

---
## Pieza A · La forma baricéntrica (semana 1)

### Celda A1 · Tu cuenta a mano, revisada por la máquina
Nodos 2, 3, 4 y f(x) = 1/x, como en la Parte I.3 de tu parcial. Haz primero la cuenta a mano en la guía (pesos w₀, w₁, w₂ y P(2,5) con la segunda forma baricéntrica). Después corre la celda.

*Nota:* para esta celda no usamos años sino los números 2, 3, 4 directamente.

In [ ]:
# Celda A1 · La cuenta de la guía
from fractions import Fraction as F
x = [F(2), F(3), F(4)]; y = [1 / xi for xi in x]; t = F(5, 2)
w = [1 / np.prod([x[j] - x[k] for k in range(3) if k != j]) for j in range(3)]
num = sum(w[j] * y[j] / (t - x[j]) for j in range(3))
den = sum(w[j] / (t - x[j]) for j in range(3))
print("pesos w_j =", [str(v) for v in w])
print("numerador =", num, "  denominador =", den, "  P(2,5) =", num / den, "=", float(num / den))
print("f(2,5) - P(2,5) =", F(2, 5) - num / den)

### Celda A2 · Cuatro maneras de evaluar el mismo polinomio
Con k años equiespaciados de Colombia, el cuaderno construye **el mismo** polinomio interpolante de cuatro maneras:

1. **Monomios en años:** P(año) = c₀ + c₁·año + … + c_{k−1}·año^{k−1}, resolviendo el sistema de Vandermonde.
2. **Monomios en s:** lo mismo, pero en la variable s ∈ [−1, 1].
3. **Lagrange con doble ciclo:** la fórmula del parcial.
4. **Segunda forma baricéntrica.**

En teoría las cuatro dan exactamente el mismo polinomio. **Predice en tu guía:** ¿cuáles coinciden en la práctica? ¿Alguna falla incluso **en los propios nodos**?

In [ ]:
# Celda A2 · Cuatro maneras
K_LISTA = [13, 21, 31]   # <-- CAMBIA ESTO si quieres

def lagrange_doble(xn, yn, t):
    P = np.zeros_like(t)
    for j in range(len(xn)):
        L = np.ones_like(t)
        for m in range(len(xn)):
            if m != j:
                L *= (t - xn[m]) / (xn[j] - xn[m])
        P += yn[j] * L
    return P

warnings.filterwarnings("ignore")
print(" k | cond(V en años) | cond(V en s) | error en los nodos: (1) (2) (3) (4)           | máx |P_i - P_bary| en 1960-2020: (1) (2) (3)")
for k in K_LISTA:
    S = equi(k); idx = (S - 1960).astype(int); yn = COL[idx]
    V1 = np.vander(S, k);     P1 = np.polyval(np.linalg.solve(V1, yn), ANOS)
    V2 = np.vander(s(S), k);  P2 = np.polyval(np.linalg.solve(V2, yn), s(ANOS))
    P3 = lagrange_doble(S, yn, ANOS); P4 = bary(S, yn, ANOS)
    en_nodos = [np.abs(P[idx] - yn).max() for P in (P1, P2, P3, P4)]
    difs = [np.abs(P - P4).max() for P in (P1, P2, P3)]
    print(f"{k:2d} | {np.linalg.cond(V1):14.1e} | {np.linalg.cond(V2):11.1e} | "
          + "  ".join(f"{e:8.1e}" for e in en_nodos) + "  | " + "  ".join(f"{d:8.1e}" for d in difs))

### Celda A3 · ¿Cuánto cuesta evaluar?
Evaluamos el interpolante con k = 61 nodos (¡todos los años!) en doscientos mil puntos. **Predice:** ¿cuántas veces más lento es el doble ciclo que la baricéntrica? Cuenta las operaciones por punto de cada fórmula antes de correr.

In [ ]:
# Celda A3 · Tiempo
k = 61; S = ANOS; yn = COL
t = np.linspace(1960, 2020, 200_000) + 1e-7     # puntos que no son nodos
w_ = pesos(S)                                       # los pesos se calculan UNA vez

t0 = time.perf_counter(); lagrange_doble(S, yn, t); t_doble = time.perf_counter() - t0
t0 = time.perf_counter()
D = s(t)[:, None] - s(S)[None, :]; Cm = w_ / D; Pb = (Cm @ yn) / Cm.sum(axis=1)
t_bary = time.perf_counter() - t0
print(f"doble ciclo: {t_doble:.2f} s    baricéntrica: {t_bary:.2f} s    razón: {t_doble / t_bary:.1f}")

---
## Pieza B · El rompecabezas de Lebesgue (semana 2)

El teorema de la guía dice que, para cualquier conjunto S de años usado como nodos,

$$E \;\le\; \max_{\text{años}} |f - P_S| \;\le\; (1 + \Lambda_S)\, E,$$

donde E es el **piso minimax discreto** (el mejor error posible con un polinomio de grado n en los 61 años) y Λ_S es la constante de Lebesgue de S. Las dos cotas se pueden **calcular**. Por eso medimos el error en "veces el piso": la razón error/E vive entre 1 y 1 + Λ.

### Celda B1 · El piso de Colombia y la cota de dos lados
**Predice en tu guía** la razón error/E de cada conjunto antes de correr.

In [ ]:
# Celda B1 · Piso y certificado para Colombia
K = 13   # <-- CAMBIA ESTO si la guía te lo pide
E = piso(COL, K - 1)
print(f"Piso minimax con {K} nodos (grado {K-1}): E = {E:.4f} M\n")
print("conjunto     | error máx (M) | error / E | Lambda  | cota 1 + Lambda | ¿se cumple?")
for nombre, S in [("equi", equi(K)), ("Chebyshev", cheb(K)), ("mock-Cheb", mock(K)),
                  ("NodoBot", NODOBOT if K == 13 else cheb(K))]:
    e = error(S, COL).max(); L = Lambda(S)
    print(f"{nombre:12s} | {e:13.4f} | {e/E:9.2f} | {L:7.2f} | {1+L:15.2f} | {E <= e <= (1+L)*E}")

### Celda B2 · Λ como amplificador de errores en los datos
Supón que **un** dato de un nodo trae un error de 0,01 M (10 000 personas). El polinomio cambia en `0,01·Lⱼ(t)`. **Predice:** ¿cuánto se mueve como máximo el polinomio de Chebyshev? ¿Y el de NodoBot? ¿En qué nodo conviene equivocarse para hacer más daño?

In [ ]:
# Celda B2 · Perturbar un dato
DELTA = 0.01   # <-- CAMBIA ESTO
for nombre, S in [("Chebyshev", cheb(13)), ("NodoBot", np.array(NODOBOT, float))]:
    S = np.array(sorted(S)); yn = COL[(S - 1960).astype(int)]
    peor = (0, None)
    for j in range(len(S)):
        yp = yn.copy(); yp[j] += DELTA
        cambio = np.abs(bary(S, yp, T_FINA) - bary(S, yn, T_FINA)).max()
        if cambio > peor[0]: peor = (cambio, int(S[j]))
    print(f"{nombre:10s}: peor cambio = {peor[0]:.4f} M (perturbando {peor[1]})   Lambda*DELTA = {Lambda(S)*DELTA:.4f} M")

### Celda B3 · ¿Y si los nodos no fueran datos?
En la demostración de la guía hay un paso que exige que los nodos estén **entre los años con dato**. Esta celda usa los nodos de Chebyshev **exactos** (1987,35, …) y los valores de una función que sí conocemos en todas partes, para que veas qué cambia. Respóndelo en la guía (B.1, pregunta sobre el paso 5).

In [ ]:
# Celda B3 · Nodos que no son datos
# Una "población" suave conocida en todo [1960, 2020]: logística
f = lambda a: 60 / (1 + np.exp(-(np.asarray(a, float) - 2000) / 18))
y_f = f(ANOS)
k = 13; i = np.arange(k); exactos = 1990 + 30 * np.cos((2 * i + 1) * np.pi / (2 * k))
E_f = piso(y_f, k - 1)
P_exactos = bary(exactos, f(exactos), ANOS)
print("nodos exactos de Chebyshev:", np.round(np.sort(exactos), 2))
print(f"piso E = {E_f:.2e}   error con nodos exactos = {np.abs(P_exactos - y_f).max():.2e}   "
      f"con años redondeados = {error(cheb(k), y_f).max():.2e}")
print(f"(1 + Lambda_exactos) * E = {(1 + lebesgue(exactos, T_FINA).max()) * E_f:.2e}")

---
## Pieza C · NodoBot de verdad (semana 3)

### Celda C1 · El entorno, escrito como en la guía
Estado: un conjunto de 13 años. Acción: mover el año número p un paso (−1 o +1), o detenerse. Recompensa: lo que **baja** el error medido en veces el piso. El verificador **enmascara** las acciones que llevarían a Λ > Λ_max. Esta celda juega un episodio con la política "toma siempre la mejor acción permitida" (es decir, la búsqueda local).

In [ ]:
# Celda C1 · Un episodio
LMAX = 5.0   # <-- CAMBIA ESTO
PAIS = "COL" # <-- CAMBIA ESTO (código de 3 letras; mira NOMBRES)
y = DATOS[PAIS]; E = piso(y, 12)
T_GRUESA = np.linspace(1960, 2020, 1201)

def acciones_permitidas(S):
    out = []
    for p in range(len(S)):
        for d in (-1, 1):
            nuevo = S[p] + d
            if 1960 <= nuevo <= 2020 and nuevo not in S:
                S2 = sorted(S[:p] + [nuevo] + S[p+1:])
                if lebesgue(np.array(S2), T_GRUESA).max() <= LMAX:   # el verificador
                    out.append((p, d, S2))
    return out

S = list(cheb(13)); retorno = 0.0; paso = 0
while True:
    actual = error(S, y).max() / E
    opciones = [(actual - error(S2, y).max() / E, p, d, S2) for p, d, S2 in acciones_permitidas(S)]
    mejor = max(opciones, default=None)
    if mejor is None or mejor[0] <= 0:
        print(f"paso {paso}: acción = DETENERSE"); break
    r, p, d, S = mejor; retorno += r; paso += 1
    print(f"paso {paso}: mover el año #{p} en {d:+d}  recompensa = {r:+.3f}  error/E = {actual - r:.3f}")
print(f"\n{NOMBRES[PAIS]}: retorno total = {retorno:.3f}   error/E final = {error(S, y).max()/E:.3f}   Lambda = {Lambda(S):.2f}")
print("años finales:", [int(a) for a in S])

### Celda C2 · Las cinco reglas en los 216 países
Para cada país calculamos error/E con: equiespaciados, Chebyshev, mock-Chebyshev, la regla voraz de AAA y NodoBot (búsqueda local sin verificador). **Predice en tu guía** la mediana de cada una. Tarda unos 30 segundos.

In [ ]:
# Celda C2 · 216 países  (≈ 30 s)
K = 13
filas = []
for c in PAISES:
    y = DATOS[c]; E = piso(y, K - 1)
    S_loc, e_loc = busqueda_local(y, K)
    S_vor = voraz(y, K)
    filas.append({"pais": c, "E": E,
                  "equi": error(equi(K), y).max() / E, "cheb": error(cheb(K), y).max() / E,
                  "mock": error(mock(K), y).max() / E, "voraz": error(S_vor, y).max() / E,
                  "nodobot": e_loc / E, "L_voraz": Lambda(S_vor), "L_nodobot": Lambda(S_loc)})
R = pd.DataFrame(filas).set_index("pais")
print(R[["equi", "cheb", "mock", "voraz", "nodobot"]].describe(percentiles=[.1, .5, .9]).round(2))
print("\nmediana de Lambda: voraz", R.L_voraz.median().round(2), "  NodoBot", R.L_nodobot.median().round(2),
      "  Chebyshev", round(Lambda(cheb(K)), 2))
print("fracción de países donde NodoBot le gana a Chebyshev:", (R.nodobot < R.cheb).mean())

### Celda C3 · ¿Se puede aprender un conjunto universal?
Partimos los países al azar en dos mitades. NodoBot-universal busca **un solo** conjunto de 13 años que minimice la mediana de error/E en la mitad de entrenamiento. Después lo probamos en la otra mitad, que nunca vio. **Predice:** ¿le gana a Chebyshev en los países nuevos?

In [ ]:
# Celda C3 · Entrenar en 108 países, probar en otros 108
SEMILLA = 0   # <-- CAMBIA ESTO para otra partición
rng = np.random.default_rng(SEMILLA); perm = rng.permutation(PAISES)
entreno, prueba = perm[:108], perm[108:]
def puntaje(S, paises): return np.median([error(S, DATOS[c]).max() / R.loc[c, "E"] for c in paises])

S = list(cheb(13)); mejor = puntaje(S, entreno)
while True:
    cand = None
    for p in range(13):
        for d in (-1, 1):
            nuevo = S[p] + d
            if nuevo < 1960 or nuevo > 2020 or nuevo in S: continue
            S2 = sorted(S[:p] + [nuevo] + S[p+1:]); v = puntaje(S2, entreno)
            if v < mejor and (cand is None or v < cand[0]): cand = (v, S2)
    if cand is None: break
    mejor, S = cand
UNIVERSAL = S
print("conjunto universal:", [int(a) for a in UNIVERSAL], "  Lambda =", round(Lambda(UNIVERSAL), 2))
for nombre, A in [("Chebyshev", cheb(13)), ("universal", UNIVERSAL)]:
    print(f"{nombre:10s}  mediana entrenamiento = {puntaje(A, entreno):.3f}   mediana prueba = {puntaje(A, prueba):.3f}")
u = np.array([error(UNIVERSAL, DATOS[c]).max() for c in prueba]); ch = np.array([error(cheb(13), DATOS[c]).max() for c in prueba])
print("en prueba, el universal le gana a Chebyshev en", round((u < ch).mean() * 100), "% de los países")
print("NodoBot país por país, mediana en prueba:", R.loc[prueba, "nodobot"].median().round(3))

### Celda C4 · El verificador: ¿cuánto cuesta exigir Λ ≤ Λ_max?
**Predice:** si NodoBot solo puede usar conjuntos con Λ ≤ 3 (más o menos la de Chebyshev), ¿conserva algo de su ventaja? Tarda alrededor de un minuto.

In [ ]:
# Celda C4 · Verificador  (≈ 1 min)
LISTA_LMAX = [3, 5, 10]   # <-- CAMBIA ESTO
violaciones = 0
for Lm in LISTA_LMAX:
    razones, lams = [], []
    for c in PAISES:
        y = DATOS[c]; E = R.loc[c, "E"]
        S, e = busqueda_local(y, 13, Lmax=Lm)
        razones.append(e / E); lams.append(Lambda(S))
        violaciones += not (1 - 1e-9 <= e / E <= 1 + lams[-1] + 1e-9)
    print(f"Lmax = {Lm:4}: mediana error/E = {np.median(razones):.3f}  percentil 90 = {np.percentile(razones, 90):.3f}  mediana Lambda = {np.median(lams):.2f}")
print("Chebyshev: mediana error/E =", R.cheb.median().round(3))
print("Casos donde falla la cota de dos lados:", violaciones)

### Celda C5 · Años que la búsqueda no vio
Tu diapositiva 29 pide "probar en datos no usados". Aquí NodoBot elige sus años mirando **solo los años pares** y lo evaluamos **solo en los impares**. **Predice** si sobrevive la ventaja. Después lee el comentario de la celda: hay una pequeña fuga en este diseño. Encuéntrala y propón cómo cerrarla (pregunta C.4 de la guía).

In [ ]:
# Celda C5 · Entrenar en años pares, evaluar en impares  (≈ 30 s)
pares = (ANOS % 2 == 0)
r_bot, r_ch = [], []
for c in PAISES:
    y = DATOS[c]; E = R.loc[c, "E"]
    S, _ = busqueda_local(y, 13, mascara=pares)
    r_bot.append(error(S, y)[~pares].max() / E); r_ch.append(error(cheb(13), y)[~pares].max() / E)
r_bot, r_ch = np.array(r_bot), np.array(r_ch)
print(f"en años impares: mediana NodoBot = {np.median(r_bot):.3f}   Chebyshev = {np.median(r_ch):.3f}   "
      f"NodoBot mejor en {np.mean(r_bot < r_ch) * 100:.0f} % de los países")
# Comentario: la búsqueda puede mover sus nodos a años impares. ¿Qué error tiene P en un nodo?

---
## Pieza E (opcional) · El mismo problema con AAA

AAA (Nakatsukasa, Sète y Trefethen, 2018) escoge sus **puntos de soporte** con la regla voraz: agrega el punto de mayor error. Pero en lugar de un polinomio construye un **cociente** en forma baricéntrica, con pesos que no son los de Lagrange. `scipy.interpolate.AAA` lo implementa.

### Celda E1 · AAA con la población de Colombia
**Predice:** con 13 puntos de soporte, ¿puede AAA quedar **por debajo del piso E** de los polinomios? ¿Qué años escoge?

In [ ]:
# Celda E1 · AAA en Colombia
from scipy.interpolate import AAA
E13 = piso(COL, 12)
for m in [7, 13, 21]:   # <-- CAMBIA ESTO
    r = AAA(s(ANOS), COL, max_terms=m, clean_up=False)
    e = np.abs(r(s(ANOS)) - COL).max()
    polos = r.poles(); reales = polos[(np.abs(polos.imag) < 1e-6) & (np.abs(polos.real) <= 1)].real
    print(f"m = {m:2d}: error en los 61 años = {e:.4f} M  (piso polinomial con 13 nodos = {E13:.4f})")
    print(f"        años de soporte = {sorted(int(v) for v in np.round(r.support_points.real * 30 + 1990))}")
    print(f"        polos reales dentro de [1960, 2020]: {np.round(reales * 30 + 1990, 2)}")

### Celda E2 · ¿Qué pasa entre un año y el siguiente?
Toma el aproximante de m = 21 y mira de cerca su primer polo real. **Predice** a qué distancia del polo hay que acercarse para que r se dispare, y por qué los 61 datos no lo notan. Después corre la parte de los 216 países.

In [ ]:
# Celda E2 · Polos entre años
r = AAA(s(ANOS), COL, max_terms=21, clean_up=False)
polos = r.poles(); reales = np.sort(polos[(np.abs(polos.imag) < 1e-6) & (np.abs(polos.real) <= 1)].real)
if len(reales):
    a = reales[0] * 30 + 1990
    print(f"polo real en el año {a:.6f}")
    for dt in [1, 0.1, 0.01, 1e-3, 1e-4, 1e-5, 1e-6]:
        print(f"r(polo + {dt:g}) = {r(s(a + dt)).real:16.3f} M")
cuenta, razones = 0, []
for c in PAISES:
    y = DATOS[c]; ra = AAA(s(ANOS), y, max_terms=13, clean_up=False); p = ra.poles()
    tiene = np.any((np.abs(p.imag) < 1e-6) & (np.abs(p.real) <= 1)); cuenta += tiene
    if not tiene: razones.append(np.abs(ra(s(ANOS)) - y).max() / R.loc[c, "E"])
print(f"\nCon 13 puntos de soporte: {cuenta} de {len(PAISES)} países tienen un polo real dentro de [1960, 2020].")
print(f"En los {len(razones)} países sin polos, mediana de error/E = {np.median(razones):.3f}")

---
### Antes de entregar
**Entorno de ejecución → Reiniciar y ejecutar todo.** Si alguna celda falla o una cifra cambia, arréglalo antes de copiarla al artículo: solo el 4 % de los cuadernos públicos de GitHub reproduce sus propios resultados guardados (Pimentel et al., 2019). Que el tuyo esté en el otro 96 % no es opcional.